# 01 - Corpus audit: source confounds and shortcut features

Runs on `domains_labelled.parquet`, before any features are built.

The threat this notebook exists to detect: **every source contributes exactly
one class.** All benign domains come from Tranco; all malicious come from
UMUDGA, URLhaus, and OpenPhish. Any artefact that distinguishes the *sources*
therefore also separates the *classes*, and a model can exploit it without
learning anything about maliciousness.

The most likely such artefact is the TLD. UMUDGA's generators emit into a
narrow set of suffixes; Tranco spans hundreds. If TLD alone approaches the
performance of the full model, the headline result is measuring dataset
assembly rather than detection.

This is the corpus-level audit. The feature-level leakage screen (single-feature
AUC, class-dependent missingness) runs later, in `03a`, once the feature matrix
exists.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
INTERIM = Path(P['data']['interim'])
df = pd.read_parquet(INTERIM/'domains_labelled.parquet')
print(df.shape)
df.head()

## 1. Source-class confound

Confirms the structural problem: each source is single-class, so `source` is a
perfect predictor and must never be used as a feature. It is not in
`features.yaml`, but this records why.

In [ ]:
ct = pd.crosstab(df['source'], df['label'])
ct['malicious_rate'] = (ct[1] / ct.sum(axis=1)).round(4) if 1 in ct else 0
display(ct)
print('Sources that are NOT single-class:',
      [s for s, r in ct['malicious_rate'].items() if 0 < r < 1] or 'none')

## 2. TLD shortcut - the critical test

Two questions:

1. How much do the TLD distributions differ by class?
2. How well does TLD *alone* classify?

The second number is the one that matters. If a TLD-only model reaches a
PR-AUC close to the full model's, the full model's headline number is
substantially explained by a dataset artefact.

In [ ]:
df['tld'] = df['domain'].str.rsplit('.', n=1).str[-1]

top = df['tld'].value_counts().head(25).index
tld_ct = pd.crosstab(df.loc[df.tld.isin(top),'tld'], df.loc[df.tld.isin(top),'label'])
tld_ct.columns = ['benign','malicious']
tld_ct['malicious_rate'] = (tld_ct.malicious/tld_ct.sum(axis=1)).round(3)
display(tld_ct.sort_values('malicious_rate', ascending=False))

print('distinct TLDs  benign   :', df[df.label==0]['tld'].nunique())
print('distinct TLDs  malicious:', df[df.label==1]['tld'].nunique())
overlap = set(df[df.label==0].tld) & set(df[df.label==1].tld)
print('TLDs present in both classes:', len(overlap))

In [ ]:
# TLD-only classifier. If this scores high, the corpus is separable by suffix.
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

s = df.sample(min(300_000, len(df)), random_state=42)
codes = s['tld'].astype('category')
X = codes.cat.codes.values.reshape(-1,1)
y = s['label'].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

import xgboost as xgb
m = xgb.XGBClassifier(max_depth=6, n_estimators=200, tree_method='hist',
                      eval_metric='aucpr', verbosity=0).fit(Xtr, ytr)
p = m.predict_proba(Xte)[:,1]
print(f'TLD-ONLY  PR-AUC {average_precision_score(yte,p):.4f}   ROC-AUC {roc_auc_score(yte,p):.4f}')
print('Interpretation: >0.90 means the corpus is largely separable by suffix alone.')

### Mitigation, if TLD-only scores high

Options, in order of preference:

1. **Restrict to shared TLDs.** Keep only domains whose TLD appears in both
   classes. Loses rows, removes the shortcut.
2. **Drop `tld` from the feature set** and report the TLD-only baseline in the
   paper as a negative control.
3. **Report both** - full feature set and TLD-excluded - as an ablation row.

Option 3 is the strongest for review: it quantifies the artefact rather than
hiding it. Whatever is chosen, record the decision below.

In [ ]:
shared = set(df[df.label==0].tld) & set(df[df.label==1].tld)
restricted = df[df.tld.isin(shared)]
print(f'full corpus      {len(df):>10,}')
print(f'shared-TLD only  {len(restricted):>10,}  ({100*len(restricted)/len(df):.1f}%)')
print(f'  benign         {(restricted.label==0).sum():>10,}')
print(f'  malicious      {(restricted.label==1).sum():>10,}')
print(f'  families       {restricted[restricted.label==1]["family"].nunique():>10}')

## 3. Length and character-set separation

DGA generators often emit fixed-length strings. If malicious domains occupy a
length range benign ones never reach, length becomes another shortcut - less
severe than TLD, since length is a legitimate signal, but worth quantifying.

In [ ]:
core = df['domain'].str.split('.').str[0]
df['core_len'] = core.str.len()
display(df.groupby('label')['core_len'].describe().round(2))

for lab, name in [(0,'benign'), (1,'malicious')]:
    v = df[df.label==lab]['core_len']
    print(f'{name:10s} p1={v.quantile(.01):.0f} p50={v.quantile(.5):.0f} p99={v.quantile(.99):.0f}')

only_mal = set(df[df.label==1].core_len.unique()) - set(df[df.label==0].core_len.unique())
print('lengths seen ONLY in malicious:', sorted(only_mal)[:20])

In [ ]:
# Per-family length uniformity: a family with near-zero variance is trivially
# separable, and inflates family-disjoint scores for the wrong reason.
fam_len = (df[df.label==1].groupby('family')['core_len']
             .agg(n='count', mean='mean', std='std').round(2)
             .sort_values('std'))
display(fam_len.head(15))
print('families with std < 0.5 (fixed-length generators):', (fam_len['std'] < 0.5).sum())

## 4. Duplicates and near-duplicates

In [ ]:
from src.evaluate import leakage
print(leakage.duplicate_audit(df, 'domain', 'label'))

# Same registrable domain reached via different sources
multi = df['domain'].duplicated().sum()
print('duplicate domain rows remaining after combine():', multi)

## 5. Record the decisions

Each finding gets an explicit decision. This text becomes the methodology
paragraph stating that a leakage audit preceded model development - which is
worth considerably more to a reviewer than a high accuracy number.

In [ ]:
entry = f'''
## Corpus audit ({pd.Timestamp.now().date()})

### Confound: source is single-class
Every source contributes exactly one class (Tranco benign; UMUDGA, URLhaus,
OpenPhish malicious). DECISION: `source` is excluded from the feature set and
is used only for reporting. Any feature that proxies for source is treated as
suspect.

### Feature: tld
TLD-only PR-AUC on a 300k sample: <FILL IN from the cell above>.
DECISION: <restrict to shared TLDs / drop tld / report as ablation>.
REASON: benign and malicious domains originate from disjoint sources, so
suffix distribution partly reflects dataset assembly rather than maliciousness.

### Feature: core_len
Fixed-length generators identified in <N> families.
DECISION: retained. Length is a legitimate detection signal, but the
family-disjoint split is reported alongside random so that any family-specific
length memorisation is visible as a generalisation gap.
'''
open(P['leakage_notes'], 'a').write(entry)
print(open(P['leakage_notes']).read()[-1200:])

---

Fill in the two `<...>` placeholders above with the actual numbers before
moving on, then run `04_split_creation`.